| Table                      | Main validation                            | Result                             |
| -------------------------- | ------------------------------------------ | ---------------------------------- |
| `raw.orders`               | Nulls, dates, duplicates, statuses         | ⚠️ Some source anomalies           |
| `raw.customers`            | Nulls, IDs, customer identity, states      | ✅                                  |
| `raw.order_items`          | Nulls, natural key, prices, item numbering | ✅                                  |
| `raw.payments`             | Nulls, payments, installments, types       | ⚠️ 2 installment anomalies         |
| `raw.reviews`              | Nulls, scores, natural key, dates, orphans | ✅                                  |
| `raw.products`             | Nulls, IDs, dimensions, categories         | ⚠️ Some source anomalies           |
| `raw.sellers`              | Nulls, IDs, states, ZIP, references        | ⚠️ ZIP/state inconsistencies       |
| `raw.geolocation`          | Nulls, coordinates, duplicates, states     | ⚠️ Duplicate/source mapping issues |
| `raw.category_translation` | Nulls, uniqueness, coverage                | ⚠️ 2 untranslated categories       |


SELECT
    COUNT(*) AS joined_rows,
    COUNT(DISTINCT o.order_id) AS orders,
    COUNT(DISTINCT c.customer_id) AS customer_records
FROM staging.orders o
JOIN staging.customers c
    ON o.customer_id = c.customer_id;

	SELECT COUNT(*) AS orphan_orders
FROM staging.orders o
LEFT JOIN staging.customers c
    ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL;


# Révision — Data Engineering & Monitoring Olist

## 1. Objectif

Construire un environnement Data Engineering autour du dataset **Olist** :

```text
Ingestion
   ↓
Transformation dbt
   ↓
Data Warehouse PostgreSQL
   ↓
Orchestration Airflow
   ↓
Monitoring Prometheus
   ↓
Visualisation Grafana
```

L'objectif est de reproduire une architecture proche d'un environnement Data réel.

---

## 2. Data Warehouse PostgreSQL

Base :

```text
olist_dw
```

Schemas principaux :

```text
raw
staging
warehouse
analytics
observability
```

### Tables Warehouse

```text
warehouse
├── dim_customers
├── dim_products
├── dim_orders
├── dim_sellers
├── dim_category
├── fact_order_items
├── fact_payments
└── fact_reviews
```

### Tables Analytics

```text
analytics
├── fct_sales
├── fct_orders
├── fct_monthly_sales
├── dim_customer_analytics
└── dim_product_analytics
```

---

## 3. dbt

dbt est utilisé pour :

* transformer les données ;
* construire les modèles analytiques ;
* effectuer les contrôles de qualité.

Résultats actuels :

```text
dbt run  → 13 modèles PASS
dbt test → 28 tests PASS
```

### Principaux tests

```text
fct_sales_not_empty
fct_sales_unique_order_item
fct_orders_unique_order
fct_sales_total_value_positive
dim_product_analytics_unique
pipeline_metrics_volume_anomaly
...
```

---

## 4. Airflow

Airflow **3.3.2** orchestre le pipeline.

DAG principal :

```text
olist_daily_pipeline
```

Planification :

```text
Tous les jours à 22:00
Timezone : Indian/Antananarivo
```

### Pipeline

```text
run_dbt
   ↓
dbt_test
   ↓
record_metrics
   ↓
dbt_observability_test
   ↓
trigger_monitoring
```

---

## 5. Observability PostgreSQL

Schema :

```text
observability
```

### `pipeline_runs`

Cette table conserve l'historique des exécutions Airflow :

```text
id
dag_id
run_id
state
start_date
end_date
duration_seconds
recorded_at
```

Elle permet notamment de suivre la durée réelle des pipelines.

Exemples de durées enregistrées :

```text
26.6 s
100.4 s
80.5 s
90.1 s
146.6 s
132.3 s
178.6 s
```

---

### `pipeline_metrics`

Cette table conserve les volumes de données produits par les pipelines :

```text
run_date
pipeline_name
table_name
row_count
```

Exemples :

```text
fct_sales
fct_orders
fct_monthly_sales
dim_customer_analytics
dim_product_analytics
```

Cela permet de comparer les volumes entre plusieurs exécutions.

---

## 6. Contrôle d'anomalie de volume

Test dbt :

```text
pipeline_metrics_volume_anomaly
```

Principe :

```text
Volume actuel
      ↓
comparaison
      ↓
Volume précédent
```

Une anomalie est détectée lorsqu'une table perd plus de **20 %** de son volume par rapport au run précédent.

Le test retourne :

```text
PASS → aucune anomalie
ERROR → anomalie détectée
```

Le résultat est également exposé à Prometheus sous forme de métrique :

```text
olist_dbt_volume_anomaly
```

avec :

```text
1 = OK
0 = anomalie
```

---

## 7. Pipeline Exporter

Nous avons développé notre propre exporter Python.

Technologies :

```text
Python
Flask
prometheus-client
psycopg
```

Architecture :

```text
PostgreSQL
     ↓
Python Pipeline Exporter
     ↓
Prometheus
```

### Métriques principales

```text
olist_pipeline_duration_seconds
olist_pipeline_fct_sales_rows
olist_pipeline_fct_orders_rows
olist_pipeline_success
olist_dbt_volume_anomaly
```

---

## 8. Prometheus

Prometheus collecte les métriques via deux exporters principaux.

```text
Prometheus
   │
   ├── postgres-exporter
   │
   └── pipeline-exporter
```

### PostgreSQL Exporter

Permet notamment de suivre :

```text
pg_database_size_bytes
```

et donc l'évolution de la taille de la base.

### Pipeline Exporter

Expose les métriques spécifiques à notre pipeline Data.

---

## 9. Grafana

Grafana utilise Prometheus comme datasource.

Dashboard :

```text
Olist Data Engineering Monitoring
```

### Panels actuels

#### Pipeline Duration

Courbe historique de la durée des pipelines :

```text
Durée
  │
  │       ╭──╮
  │   ╭───╯  ╰──╮
  │───╯         ╰──
  └──────────────────→ Temps
```

Cette courbe utilise les durées réelles enregistrées dans PostgreSQL.

---

#### fct_sales Row Count

Permet de visualiser l'évolution du nombre de lignes de `fct_sales`.

---

#### Database Size

Suit la taille de la base PostgreSQL dans le temps.

---

#### Backend Connections

Permet de surveiller les connexions PostgreSQL.

---

#### Pipeline Status

Métrique :

```text
olist_pipeline_success
```

Mapping :

```text
1 → UP
0 → DOWN
```

---

#### DBT Volume Anomaly

Métrique :

```text
olist_dbt_volume_anomaly
```

Mapping :

```text
1 → OK
0 → Anomaly
```

Le panneau fonctionne correctement avec une période comme :

```text
Last 1 hour
```

---

## 10. Démonstration historique

Nous avons également créé une table de démonstration :

```text
observability.pipeline_metrics_demo
```

Elle contient plusieurs valeurs historiques de `fct_sales`.

Exemple :

```text
108000
111500
109800
115200
113700
118400
116900
121500
119800
124000
```

Cela nous a permis de comprendre le fonctionnement d'une véritable série temporelle :

```text
PostgreSQL
     ↓
Exporter
     ↓
Prometheus
     ↓
Historique des métriques
     ↓
Grafana Time Series
```

---

## 11. UP / DOWN Timeline

Nous avons commencé à réfléchir à une timeline UP/DOWN.

La métrique actuelle :

```text
olist_pipeline_success
```

représente l'état du **dernier run connu**.

Elle permet donc d'afficher :

```text
UP
DOWN
```

Mais elle ne représente pas encore parfaitement les différentes phases d'une exécution.

Pour une vraie timeline d'exécution, on pourrait aller vers :

```text
RUNNING
   ↓
SUCCESS
```

ou :

```text
RUNNING
   ↓
FAILED
```

Les informations nécessaires existent déjà dans :

```text
observability.pipeline_runs
```

notamment :

```text
start_date
end_date
state
duration_seconds
```

---

# 12. Architecture globale actuelle

```text
                       OLIST CSV
                          │
                          ↓
                 ┌─────────────────┐
                 │   PostgreSQL    │
                 │ raw / staging   │
                 └────────┬────────┘
                          │
                          ↓
                         dbt
                          │
                          ↓
                 ┌─────────────────┐
                 │    Analytics    │
                 │    Warehouse    │
                 └────────┬────────┘
                          │
                          ↓
                       Airflow
                          │
                          ↓
                 ┌─────────────────┐
                 │  Observability  │
                 │   PostgreSQL    │
                 └────────┬────────┘
                          │
                          ↓
                Pipeline Exporter
                          │
                          ↓
                    Prometheus
                          │
                          ↓
                     Grafana
```

---

# 13. Ce qu'on sait maintenant faire

À ce stade, le projet couvre :

* [x] ingestion des données Olist ;
* [x] stockage PostgreSQL ;
* [x] modélisation Data Warehouse ;
* [x] transformation avec dbt ;
* [x] tests de qualité dbt ;
* [x] orchestration avec Airflow ;
* [x] historique des runs ;
* [x] collecte des volumes ;
* [x] détection d'anomalies de volume ;
* [x] création d'un exporter Prometheus personnalisé ;
* [x] monitoring PostgreSQL ;
* [x] monitoring du pipeline ;
* [x] visualisation historique dans Grafana ;
* [x] suivi de la durée des pipelines ;
* [x] suivi des volumes ;
* [x] statut UP/DOWN du dernier pipeline.

---

# 14. Point suivant

La prochaine amélioration logique du monitoring est :

```text
UP / DOWN Timeline
```

avec une distinction plus précise entre :

```text
RUNNING
SUCCESS
FAILED
```

en utilisant l'historique déjà présent dans :

```text
observability.pipeline_runs
```

Puis nous pourrons passer à un autre sujet Data Engineering.


# Olist Data Engineering — Fiche technique

* **OS** : Ubuntu 26.04 LTS sous WSL

* **Python** : 3.14.4 + `.venv`

* **PostgreSQL** : 18.6, base `olist_dw`, installé sur WSL

  * Port `5432`
  * Utilisateur `dev`
  * Schemas : `raw`, `staging`, `warehouse`, `analytics`, `observability`

* **Docker** : utilisé pour isoler les composants Data et Monitoring

  * Communication entre containers via Docker network
  * Accès au PostgreSQL WSL via `host.docker.internal`

* **Airflow** : 3.3.2, tourne sur **Docker Compose**, orchestre le pipeline quotidien

  * DAG principal : `olist_daily_pipeline`
  * Exécution quotidienne à `22:00` (`Indian/Antananarivo`)
  * Enchaînement : `dbt run → dbt test → record_metrics → dbt observability test`

* **dbt** : tourne dans un **container Docker**, transformations + Data Quality

  * `staging` → Views
  * `marts` → Tables
  * Actuellement : `13 models` + `28 tests` PASS

* **Data Warehouse** : PostgreSQL

  * Modèle dimensionnel : `dimensions + facts`
  * Schemas séparés pour les différentes étapes du pipeline

* **Observability** : schema PostgreSQL dédié

  * `pipeline_runs` → historique des exécutions Airflow
  * `pipeline_metrics` → historique des volumes par table et par run
  * Permet de conserver les données nécessaires au monitoring

* **Data Quality** : tests dbt

  * Unicité, données non vides, valeurs positives, cohérence métier
  * Détection d'une baisse de volume supérieure à `20 %`

* **Pipeline Exporter** : Python + Flask + `prometheus-client` + `psycopg`

  * Exporter développé spécifiquement pour notre pipeline
  * Endpoint `/metrics` sur le port `8000`
  * Interroge directement PostgreSQL `observability`

* **PostgreSQL Exporter** : tourne sur Docker

  * `prometheuscommunity/postgres-exporter`
  * Expose les métriques techniques PostgreSQL

* **Prometheus** : tourne sur Docker

  * Scrape les exporters toutes les `15 s`
  * Conserve l'historique temporel des métriques

* **Grafana** : tourne sur Docker

  * Datasource : Prometheus
  * Dashboard : `Olist Data Engineering Monitoring`
  * Time Series pour les évolutions, Stat pour les états

* **Métriques principales** :

  * `olist_pipeline_duration_seconds` → durée du dernier run
  * `olist_pipeline_fct_sales_rows` → volume `fct_sales`
  * `olist_pipeline_fct_orders_rows` → volume `fct_orders`
  * `olist_pipeline_success` → état du dernier run
  * `olist_dbt_volume_anomaly` → état du contrôle de volume

* **Monitoring flow** :

  * `PostgreSQL → Exporter → Prometheus → Grafana`
  * PostgreSQL contient les données métier et l'historique d'observabilité

* **Pipeline flow** :

  * `Olist → PostgreSQL → dbt → Airflow → Observability → Prometheus → Grafana`

* **UP / DOWN actuel** :

  * `1 = UP / success`
  * `0 = DOWN / failure`
  * Basé sur le dernier run connu
  * Évolution prévue : vraie timeline `RUNNING / SUCCESS / FAILED`


le port 8000 est-il libre ?

Exécute :

ss -lntp | grep :8000


Prometheus — ce qu'on vient de faire
Création du service pipeline-exporter
Expose les métriques personnalisées de notre pipeline Airflow.
Fichiers :
monitoring/pipeline-exporter/app.py
monitoring/pipeline-exporter/Dockerfile
monitoring/pipeline-exporter/requirements.txt
Création de l'image Docker

Commande :

docker build -t olist-pipeline-exporter:1.0 ~/data-learning/monitoring/pipeline-exporter
Test de l'exporter

Health check :

curl http://localhost:8000/health

Métriques :

curl http://localhost:8000/metrics
Intégration de pipeline-exporter dans Docker Compose

Ajout du service pipeline-exporter dans :

monitoring/prometheus/docker-compose.yml

Vérification :

docker compose config --services
Configuration de Prometheus

Ajout du job :

- job_name: "olist-pipeline"
  static_configs:
    - targets:
        - "pipeline-exporter:8000"

Fichier :

monitoring/prometheus/prometheus.yml
Reconstruction et démarrage des services

Commande :

docker compose up -d --build
Vérification du réseau Docker

Prometheus et olist-pipeline-exporter sont sur :

prometheus_default
Test de communication Prometheus → Exporter

Commande :

docker exec prometheus wget -qO- http://pipeline-exporter:8000/metrics | grep olist_pipeline

Résultat :

olist_pipeline_duration_seconds 90.149598
Redémarrage de Prometheus après modification de configuration

Commande :

docker compose restart prometheus
Validation finale dans Prometheus
Target olist-pipeline : UP

Requête PromQL :

olist_pipeline_duration_seconds

Résultat :

90.149598

Résultat : Prometheus récupère maintenant automatiquement la durée du dernier pipeline Airflow depuis PostgreSQL via notre pipeline-exporter.